# Crop Recommendation Analysis — Problem Analysis Workshop A
## A public-sector exploratory report

| Team member | Student ID | Lead responsibility |
|---|---|---|
| [Member 1 name] | [Member 1 ID] | Data preparation and statistical profiling |
| [Member 2 name] | [Member 2 ID] | Visual analysis and normality |
| [Member 3 name] | [Member 3 ID] | Mean comparison and communication |



**Analysis reference:** `assignment 1.ipynb`. Every analysis section is carried forward and adapted; this workshop adds normality, variance, and mean comparisons. 

## 1. Project Overview & Use Case Summary

**Question:** How do growing conditions vary in this dataset, and how do rainfall values associated with rice and maize differ in distribution, variability, and mean?

**Proposed audience:** A regional agricultural planning committee comprising public-sector analysts, agricultural extension officers, and farmer-association representatives. T

**Expected outcome:** Identify patterns worth investigating for water availability and crop planning, then specify the local evidence needed before policy action.

**Finding at a glance:** Rice-labelled records have mean rainfall **236.18 mm**, compared with **84.77 mm** for maize-labelled records, a difference of **151.41 mm**. Both groups reject normality at 5%. These are associations in a crop recommendation dataset, not measured irrigation requirements or causal crop effects.



## 2. Research Question

**Original question retained as context:** “Which soil composition and environmental factors are most strongly associated with specific recommended crop types?”

**Question answered by this lab:** What soil and climate patterns are visible across crop labels, and how do rainfall distributions, variances, and means differ between rice and maize?

The broad question motivates the exploratory sections; the current work does not rank predictive importance or build a recommendation model.

| Original planning question | Adaptation for this lab |
|---|---|
| Why this dataset? | Seven soil/climate measurements and crop labels support descriptive comparison and a fixed group-comparison exercise. |
| One-time or ongoing? | One-time exploratory lab report. Ongoing decision support would require current local data, repeated validation, and a defined update process. |
| Major or minor decision? | Real crop choices can have major consequences. This classroom report frames questions rather than approving farm decisions. |
| Broad use or individual use? | A general teaching dataset; it lacks field location, dates, soil texture, seasons, and farm history for individualized advice. |

## 3. Methodology & Notebook Structure

The analysis retains every section of `assignment 1.ipynb`: project overview, research question, methodology, ingestion and cleansing, object-oriented statistical profiling, all four visual types, and the discussion. It then adds the new lab deliverables.

1. Configure libraries and load the unchanged supplied data; include a source-download verification option.
2. Validate data and create a descriptively renamed analysis table without rounding the underlying values.
3. Use `CropAnalyzer` for central tendency, dispersion, quartiles, and numerical summaries.
4. Recreate box-whisker, rainfall histogram/KDE, temperature–humidity scatter, and three-set Venn views.
5. Discuss verified exploratory findings, then expand histograms to all numerical variables.
6. Assess rainfall normality within rice and maize using QQ plots and Shapiro–Wilk.
7. Compare the same variable's variances using a two-sided F-test and a median-centered Levene check.
8. Compare means using Welch's t-statistic, report required interpretations, and communicate limitations to the proposed audience.

Detailed responsibilities and the reference-to-new-section mapping are in `team plan.md`.

## 4. Setup and Configuration

Use Python 3.12 or later. Keep this notebook beside `requirements.txt` and the `data` folder. Install the packages once using the commented cell below, then restart the kernel and select **Run All**. A working internet connection is needed only for installation or source verification; the packaged CSV supports offline use.

The executed report uses the supplied data and fixed rice/maize comparison. If the data or groups change, rerun the code and revise the written interpretations and Whiteboard to match. No records are removed to improve a p-value.

In [1]:
# Run this line once if packages are missing, then restart the kernel.
# %pip install -r requirements.txt

In [2]:
from pathlib import Path
from io import BytesIO
from urllib.request import Request, urlopen
import zipfile
import hashlib
import platform
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib_venn import venn3
from IPython.display import display

ALPHA = 0.05
GROUP_A, GROUP_B = 'rice', 'maize'
NUMERIC = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
DATA_PATH = Path('data/Crop_recommendation.csv')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
COLORS = {'rice': '#167D9A', 'maize': '#D77B32'}
LABELS = {'N': 'Nitrogen (source ratio)', 'P': 'Phosphorus (source ratio)',
          'K': 'Potassium (source ratio)', 'temperature': 'Temperature (°C)',
          'humidity': 'Relative humidity (%)', 'ph': 'Soil pH', 'rainfall': 'Rainfall (mm)'}
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 170,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.precision', 4)
print('Python:', platform.python_version())
display(pd.Series({p: importlib.metadata.version(p) for p in
                   ['numpy', 'pandas', 'scipy', 'matplotlib', 'seaborn', 'matplotlib-venn']}, name='Version'))

def save_show(fig, filename):
    fig.savefig(FIG_DIR / filename, bbox_inches='tight', facecolor='white')
    plt.show()

Python: 3.14.7


numpy               2.5.3
pandas              3.0.5
scipy              1.18.1
matplotlib         3.11.2
seaborn            0.13.2
matplotlib-venn     1.1.2
Name: Version, dtype: str

## 5. Source, Download, and Data Dictionary

The analysis uses the user-supplied CSV unchanged. An independent download from the [Kaggle Crop Recommendation Dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset) was compared with it: all parsed rows and columns matched. The notebook includes that download-and-compare step below.

The [dataset mirror documentation](https://github.com/DataWithVishal/CropRecommendation) describes augmentation of rainfall, climate, and fertilizer data associated with India. Its field definitions identify temperature in °C, humidity in percent, rainfall in mm, and N/P/K as soil nutrient ratios. This documentation does not establish the rainfall accumulation period, individual record locations, independent field sampling, or local applicability. Do not interpret rainfall as monthly or annual. Do not assign kg/ha to the nutrient columns without stronger metadata. A mirror's repository license does not independently establish the dataset license; check the publisher's terms before wider redistribution.

| Field | Interpretation used here |
|---|---|
| N, P, K | Soil nitrogen, phosphorus, and potassium ratios as described by the source |
| temperature | Temperature (°C) |
| humidity | Relative humidity (%) |
| ph | Soil pH |
| rainfall | Rainfall (mm); accumulation period unspecified |
| label | Crop category associated with each record |

The supplied CSV remains the analysis input even when source verification runs. The default below supports offline reruns; set `VERIFY_SOURCE_ONLINE = True` to repeat the download. Source verification was run during preparation of this report.

In [3]:
DATA_PATH = Path('Crop_recommendation.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('CSV file not found in the project root: Crop_recommendation.csv')

df = pd.read_csv(DATA_PATH)
print('Loaded dataset shape:', df.shape)
display(df.head())

Loaded dataset shape: (2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.8797,82.0027,6.5030,202.9355,rice
1,85,58,41,21.7705,80.3196,7.0381,226.6555,rice
2,60,55,44,23.0045,82.3208,7.8402,263.9642,rice
3,74,35,40,26.4911,80.1584,6.9804,242.8640,rice
4,78,42,42,20.1302,81.6049,7.6285,262.7173,rice


## 6. Data Ingestion & Cleansing

Check schema, types, finite values, missing entries, duplicates, crop counts, and broad physical bounds. These checks establish internal consistency, not measurement accuracy. Keep all original observations unless a documented error is found.

The reference renaming step is retained in Section 7. Unlike the reference, we do not round before testing or remove rows simply because rounding makes them identical. No exact duplicate rows were found, so no deduplication is necessary.

In [4]:
assert list(df.columns) == NUMERIC + ['label'], 'Unexpected columns'
assert all(pd.api.types.is_numeric_dtype(df[col]) for col in NUMERIC)
quality = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(),
                        'unique_values': df.nunique()})
display(quality)
bounds = pd.Series({
    'Nonfinite numeric entries': int((~np.isfinite(df[NUMERIC])).sum().sum()),
    'Negative nutrient values': int((df[['N', 'P', 'K']] < 0).sum().sum()),
    'Humidity outside 0–100': int((~df.humidity.between(0, 100)).sum()),
    'pH outside 0–14': int((~df.ph.between(0, 14)).sum()),
    'Negative rainfall values': int((df.rainfall < 0).sum()),
    'Exact duplicate rows': int(df.duplicated().sum())
}, name='Flag count')
display(bounds)
assert df.isna().sum().sum() == 0 and bounds.sum() == 0
counts = df.label.value_counts().sort_index()
display(counts.to_frame('Records'))
print(f'{len(df):,} records, {df.label.nunique()} crop labels; all observations retained.')

,dtype,missing,unique_values
N,int64,0,137
P,int64,0,117
K,int64,0,73
temperature,float64,0,2200
humidity,float64,0,2200
ph,float64,0,2200
rainfall,float64,0,2200
label,str,0,22


Nonfinite numeric entries    0
Negative nutrient values     0
Humidity outside 0–100       0
pH outside 0–14              0
Negative rainfall values     0
Exact duplicate rows         0
Name: Flag count, dtype: int64

,Records
label,
apple,100
banana,100
blackgram,100
chickpea,100
coconut,100
coffee,100
cotton,100
grapes,100
jute,100


2,200 records, 22 crop labels; all observations retained.


**Assessment:** All 2,200 rows are retained. There are seven numeric features and one categorical label, with 22 labels and 100 rows each. No missing values, exact duplicates, nonfinite values, or violations of the listed bounds were found. There are no farm identifiers or dates with which to check repeated measurements. Balanced labels must not be interpreted as equal cultivated areas or equal food supply.

## 7. Statistical Profiling & Object-Oriented Analysis

This section adapts the `CropAnalyzer` class from `assignment 1.ipynb`. It retains central tendency, sample variance, sample standard deviation, quartile boundaries, detailed profiles, and the complete numerical summary.

**Precision correction:** Descriptive column names are retained in `analysis_df`, a renamed copy of the validated raw data. Full precision is preserved for all calculations and set membership. Round only the displayed results. This avoids creating artificial ties or changing thresholds. The inferential sections continue using `df`, with original column names; both tables contain identical measurements.

**Mode correction:** When every value occurs once, report “No repeated value” rather than selecting the smallest value as a misleading unique mode. If several values share the highest frequency, report the tie. Quartiles divide data into four parts; Q1, Q2, and Q3 are the internal boundaries. The source's Q4 convention is shown explicitly as the 100th percentile/maximum.

In [5]:
RENAME = {'N': 'Nitrogen', 'P': 'Phosphorus', 'K': 'Potassium',
          'temperature': 'Temperature_C', 'humidity': 'Humidity_Pct',
          'ph': 'pH', 'rainfall': 'Rainfall_mm', 'label': 'Crop'}
analysis_df = df.rename(columns=RENAME).copy()
display(analysis_df.head(10).round(2))

class CropAnalyzer:
    """Summarize recorded soil and climate values without altering measurements."""
    def __init__(self, data):
        self.df = data.copy()
        self.numeric_cols = self.df.select_dtypes(include='number').columns.tolist()

    @staticmethod
    def mode_description(series):
        counts = series.value_counts()
        if counts.empty:
            return 'No observations'
        if counts.iloc[0] == 1:
            return 'No repeated value'
        modes = counts[counts == counts.iloc[0]].index.tolist()
        if len(modes) == 1:
            return f'{modes[0]:.2f} (frequency {counts.iloc[0]})'
        shown = ', '.join(f'{x:.2f}' for x in sorted(modes)[:4])
        return f'{len(modes)} tied modes: {shown}; frequency {counts.iloc[0]}'

    def numerical_summary(self):
        rows = []
        for name in self.numeric_cols:
            s = self.df[name].dropna()
            rows.append({'Variable': name, 'n': len(s), 'Mean': s.mean(),
                         'Median / Q2': s.median(), 'Mode': self.mode_description(s),
                         'Sample variance': s.var(ddof=1), 'Sample SD': s.std(ddof=1),
                         'Minimum': s.min(), 'Q1 (25%)': s.quantile(.25),
                         'Q3 (75%)': s.quantile(.75), 'Q4 / maximum (100%)': s.max()})
        return pd.DataFrame(rows).set_index('Variable')

    def calculate_center_and_spread(self, column_name):
        if column_name not in self.numeric_cols:
            raise ValueError('Choose a numeric column from the analysis table.')
        profile = self.numerical_summary().loc[[column_name]]
        display(profile.round(2).T)
        return profile

    def display_numerical_summary(self):
        summary = self.numerical_summary()
        display(summary.round(2))
        return summary

analyzer = CropAnalyzer(analysis_df)
nitrogen_profile = analyzer.calculate_center_and_spread('Nitrogen')
rainfall_profile = analyzer.calculate_center_and_spread('Rainfall_mm')
complete_summary = analyzer.display_numerical_summary()
assert np.isclose(complete_summary.loc['Rainfall_mm', 'Mean'], df.rainfall.mean())

,Nitrogen,Phosphorus,Potassium,Temperature_C,Humidity_Pct,pH,Rainfall_mm,Crop
0,90,42,43,20.88,82.00,6.50,202.94,rice
1,85,58,41,21.77,80.32,7.04,226.66,rice
2,60,55,44,23.00,82.32,7.84,263.96,rice
3,74,35,40,26.49,80.16,6.98,242.86,rice
4,78,42,42,20.13,81.60,7.63,262.72,rice
5,69,37,42,23.06,83.37,7.07,251.05,rice
6,69,55,38,22.71,82.64,5.70,271.32,rice
7,94,53,40,20.28,82.89,5.72,241.97,rice
8,89,54,38,24.52,83.54,6.69,230.45,rice
9,68,58,38,23.22,83.03,6.34,221.21,rice


Variable,Nitrogen
n,2200
Mean,50.55
Median / Q2,37.0
Mode,"2 tied modes: 22.00, 40.00; frequency 44"
Sample variance,1362.89
Sample SD,36.92
Minimum,0.0
Q1 (25%),21.0
Q3 (75%),84.25
Q4 / maximum (100%),140.0


Variable,Rainfall_mm
n,2200
Mean,103.46
Median / Q2,94.87
Mode,No repeated value
Sample variance,3020.42
Sample SD,54.96
Minimum,20.21
Q1 (25%),64.55
Q3 (75%),124.27
Q4 / maximum (100%),298.56


,n,Mean,Median / Q2,Mode,Sample variance,Sample SD,Minimum,Q1 (25%),Q3 (75%),Q4 / maximum (100%)
Variable,,,,,,,,,,
Nitrogen,2200,50.55,37.00,"2 tied modes: 22.00, 40.00; frequency 44",1362.89,36.92,0.00,21.00,84.25,140.00
Phosphorus,2200,53.36,51.00,60.00 (frequency 56),1088.07,32.99,5.00,28.00,68.00,145.00
Potassium,2200,48.15,32.00,17.00 (frequency 90),2565.21,50.65,5.00,20.00,49.00,205.00
Temperature_C,2200,25.62,25.60,No repeated value,25.64,5.06,8.83,22.77,28.56,43.68
Humidity_Pct,2200,71.48,80.47,No repeated value,495.68,22.26,14.26,60.26,89.95,99.98
pH,2200,6.47,6.43,No repeated value,0.60,0.77,3.50,5.97,6.92,9.94
Rainfall_mm,2200,103.46,94.87,No repeated value,3020.42,54.96,20.21,64.55,124.27,298.56
